# US Census Data Discovery and storing custom census default variables

This example notebook covers the Census Data Discovery tools in the `censusdc` package and provides a short tutorial on how to store and load a custom set of defaults for census data products

First start by importing some necessary packages

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

### Census Data discovery

The census data discovery tools can be imported from the main package like so:

In [2]:
from censusdc import data_discovery

### Get information about supported census products

The `get_supported_products()` method queries the US Census Bureau API and returns a pandas dataframe of Census API products that the Census data collector currently supports.

This dataframe contains information about the vintage, dataset information, and API links to the geographies and variables

In [3]:
dfsupport = data_discovery.get_supported_products()
dfsupport.head(10)

,vintage,dataset,title,description,geographyLink,variablesLink
0,2010,acs-acs1,ACS 1-Year Detailed Tables,The American Community Survey (ACS) is a natio...,http://api.census.gov/data/2010/acs/acs1/geogr...,http://api.census.gov/data/2010/acs/acs1/varia...
1,2010,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2010/acs/acs5/geogr...,http://api.census.gov/data/2010/acs/acs5/varia...
2,2010,acs-acs5-profile,ACS 5-Year Data Profiles,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2010/acs/acs5/profi...,http://api.census.gov/data/2010/acs/acs5/profi...
3,2010,dec-sf1,Decennial SF1,Summary File 1 (SF 1) contains detailed tables...,http://api.census.gov/data/2010/dec/sf1/geogra...,http://api.census.gov/data/2010/dec/sf1/variab...
4,2011,acs-acs1,ACS 1-Year Detailed Tables,The American Community Survey (ACS) is a natio...,http://api.census.gov/data/2011/acs/acs1/geogr...,http://api.census.gov/data/2011/acs/acs1/varia...
5,2011,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2011/acs/acs5/geogr...,http://api.census.gov/data/2011/acs/acs5/varia...
6,2011,acs-acs5-profile,ACS 5-Year Data Profiles,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2011/acs/acs5/profi...,http://api.census.gov/data/2011/acs/acs5/profi...
7,2012,acs-acs1,ACS 1-Year Detailed Tables,The American Community Survey (ACS) is a natio...,http://api.census.gov/data/2012/acs/acs1/geogr...,http://api.census.gov/data/2012/acs/acs1/varia...
8,2012,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2012/acs/acs5/geogr...,http://api.census.gov/data/2012/acs/acs5/varia...
9,2012,acs-acs5-profile,ACS 5-Year Data Profiles,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2012/acs/acs5/profi...,http://api.census.gov/data/2012/acs/acs5/profi...


This dataframe can also be filtered by, for example let's get look at only the American Community Survey: 5-Year Estimates products

In [4]:
dfacs5 = dfsupport[dfsupport["dataset"] == "acs-acs5"]
dfacs5

,vintage,dataset,title,description,geographyLink,variablesLink
1,2010,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2010/acs/acs5/geogr...,http://api.census.gov/data/2010/acs/acs5/varia...
5,2011,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2011/acs/acs5/geogr...,http://api.census.gov/data/2011/acs/acs5/varia...
8,2012,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2012/acs/acs5/geogr...,http://api.census.gov/data/2012/acs/acs5/varia...
11,2013,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2013/acs/acs5/geogr...,http://api.census.gov/data/2013/acs/acs5/varia...
14,2014,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2014/acs/acs5/geogr...,http://api.census.gov/data/2014/acs/acs5/varia...
17,2015,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2015/acs/acs5/geogr...,http://api.census.gov/data/2015/acs/acs5/varia...
20,2016,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2016/acs/acs5/geogr...,http://api.census.gov/data/2016/acs/acs5/varia...
23,2017,acs-acs5,ACS 5-Year Detailed Tables,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2017/acs/acs5/geogr...,http://api.census.gov/data/2017/acs/acs5/varia...
29,2018,acs-acs5,American Community Survey: 5-Year Estimates: D...,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2018/acs/acs5/geogr...,http://api.census.gov/data/2018/acs/acs5/varia...
36,2009,acs-acs5,American Community Survey: 5-Year Estimates: D...,The American Community Survey (ACS) is an ongo...,http://api.census.gov/data/2009/acs/acs5/geogr...,http://api.census.gov/data/2009/acs/acs5/varia...


### Checking the supported geographies for a dataset

The `get_geographies()` method queries the census API and returns to the user Census Data Collector supported geographies. This method takes the following parameters:
   - dataset : a dataset string like "acs-acs5" or a list of datasets
   - year : year or a list of years

Lets get supported geographies for a single data product

In [5]:
dfgeo = data_discovery.get_geographies("acs-acs5", 2021)
dfgeo.head(10)

,dataset,year,name,geoLevelDisplay,referenceDate,requires,wildcard,optionalWithWCFor
0,acs-acs5,2021,us,010,2021-01-01,NaN,NaN,NaN
1,acs-acs5,2021,region,020,2021-01-01,NaN,NaN,NaN
2,acs-acs5,2021,division,030,2021-01-01,NaN,NaN,NaN
3,acs-acs5,2021,state,040,2021-01-01,NaN,NaN,NaN
4,acs-acs5,2021,county,050,2021-01-01,[state],[state],state
5,acs-acs5,2021,county subdivision,060,2021-01-01,"[state, county]",[county],county
6,acs-acs5,2021,subminor civil division,067,2021-01-01,"[state, county, county subdivision]",NaN,NaN
7,acs-acs5,2021,place/remainder (or part),070,2021-01-01,"[state, county, county subdivision]",NaN,NaN
8,acs-acs5,2021,tract,140,2021-01-01,"[state, county]",[county],county
9,acs-acs5,2021,block group,150,2021-01-01,"[state, county, tract]","[county, tract]",tract


### Getting a listing of available variables from a census product

The `get_variables` method pulls a listing of all available variables from the U.S Census API tables for a requested product. The method takes the following parameters:
   - `dataset`: a dataset string like "acs-acs5"
   - `year`: the year (int)

In [8]:
acs_vars = data_discovery.get_variables("acs-acs5", 2021)
acs_vars.head(20)

,dataset,year,name,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,acs-acs5,2021,AIANHH,Geography,NaN,NaN,N/A,0,NaN,NaN,NaN,NaN
1,acs-acs5,2021,AIHHTL,Geography,NaN,NaN,N/A,0,NaN,NaN,NaN,NaN
2,acs-acs5,2021,AIRES,Geography,NaN,NaN,N/A,0,NaN,NaN,NaN,NaN
3,acs-acs5,2021,ANRC,Geography,NaN,NaN,N/A,0,NaN,NaN,NaN,NaN
4,acs-acs5,2021,B01001A_001E,Estimate!!Total:,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_001EA,B01001A_001M,B01001A_001MA",NaN
5,acs-acs5,2021,B01001A_002E,Estimate!!Total:!!Male:,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_002EA,B01001A_002M,B01001A_002MA",NaN
6,acs-acs5,2021,B01001A_003E,Estimate!!Total:!!Male:!!Under 5 years,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_003EA,B01001A_003M,B01001A_003MA",NaN
7,acs-acs5,2021,B01001A_004E,Estimate!!Total:!!Male:!!5 to 9 years,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_004EA,B01001A_004M,B01001A_004MA",NaN
8,acs-acs5,2021,B01001A_005E,Estimate!!Total:!!Male:!!10 to 14 years,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_005EA,B01001A_005M,B01001A_005MA",NaN
9,acs-acs5,2021,B01001A_006E,Estimate!!Total:!!Male:!!15 to 17 years,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_006EA,B01001A_006M,B01001A_006MA",NaN


In [7]:
print(len(acs_vars))

27927


Because this returns a whole bunch of variables we can filter through it to find things we might want using pandas

In [8]:
tot_vars = acs_vars[acs_vars['label'] == "Estimate!!Total:"]
tot_vars

,dataset,year,name,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
4,acs-acs5,2021,B01001A_001E,Estimate!!Total:,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_001EA,B01001A_001M,B01001A_001MA",NaN
35,acs-acs5,2021,B01001B_001E,Estimate!!Total:,SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE),int,B01001B,0,NaN,NaN,"B01001B_001EA,B01001B_001M,B01001B_001MA",NaN
66,acs-acs5,2021,B01001C_001E,Estimate!!Total:,SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ...,int,B01001C,0,NaN,NaN,"B01001C_001EA,B01001C_001M,B01001C_001MA",NaN
97,acs-acs5,2021,B01001D_001E,Estimate!!Total:,SEX BY AGE (ASIAN ALONE),int,B01001D,0,NaN,NaN,"B01001D_001EA,B01001D_001M,B01001D_001MA",NaN
128,acs-acs5,2021,B01001E_001E,Estimate!!Total:,SEX BY AGE (NATIVE HAWAIIAN AND OTHER PACIFIC ...,int,B01001E,0,NaN,NaN,"B01001E_001EA,B01001E_001M,B01001E_001MA",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
27713,acs-acs5,2021,C27014_001E,Estimate!!Total:,PUBLIC HEALTH INSURANCE BY WORK EXPERIENCE,int,C27014,0,NaN,NaN,"C27014_001EA,C27014_001M,C27014_001MA",NaN
27723,acs-acs5,2021,C27016_001E,Estimate!!Total:,HEALTH INSURANCE COVERAGE STATUS BY RATIO OF I...,int,C27016,0,NaN,NaN,"C27016_001EA,C27016_001M,C27016_001MA",NaN
27774,acs-acs5,2021,C27017_001E,Estimate!!Total:,PRIVATE HEALTH INSURANCE BY RATIO OF INCOME TO...,int,C27017,0,NaN,NaN,"C27017_001EA,C27017_001M,C27017_001MA",NaN
27825,acs-acs5,2021,C27018_001E,Estimate!!Total:,PUBLIC HEALTH INSURANCE BY RATIO OF INCOME TO ...,int,C27018,0,NaN,NaN,"C27018_001EA,C27018_001M,C27018_001MA",NaN


We can also use conecpts or data groups to filter data further

In [9]:
tot_vars.concept.unique()

<ArrowStringArray>
[                                                                         'SEX BY AGE (WHITE ALONE)',
                                                      'SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE)',
                                              'SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ALONE)',
                                                                          'SEX BY AGE (ASIAN ALONE)',
                                     'SEX BY AGE (NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE)',
                                                                'SEX BY AGE (SOME OTHER RACE ALONE)',
                                                                    'SEX BY AGE (TWO OR MORE RACES)',
                                                  'SEX BY AGE (WHITE ALONE, NOT HISPANIC OR LATINO)',
                                                                   'SEX BY AGE (HISPANIC OR LATINO)',
                                                               

In [10]:
# filter by concept
sxage = tot_vars[tot_vars["concept"].str.contains("SEX BY AGE")]
sxage

,dataset,year,name,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
4,acs-acs5,2021,B01001A_001E,Estimate!!Total:,SEX BY AGE (WHITE ALONE),int,B01001A,0,NaN,NaN,"B01001A_001EA,B01001A_001M,B01001A_001MA",NaN
35,acs-acs5,2021,B01001B_001E,Estimate!!Total:,SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE),int,B01001B,0,NaN,NaN,"B01001B_001EA,B01001B_001M,B01001B_001MA",NaN
66,acs-acs5,2021,B01001C_001E,Estimate!!Total:,SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ...,int,B01001C,0,NaN,NaN,"B01001C_001EA,B01001C_001M,B01001C_001MA",NaN
97,acs-acs5,2021,B01001D_001E,Estimate!!Total:,SEX BY AGE (ASIAN ALONE),int,B01001D,0,NaN,NaN,"B01001D_001EA,B01001D_001M,B01001D_001MA",NaN
128,acs-acs5,2021,B01001E_001E,Estimate!!Total:,SEX BY AGE (NATIVE HAWAIIAN AND OTHER PACIFIC ...,int,B01001E,0,NaN,NaN,"B01001E_001EA,B01001E_001M,B01001E_001MA",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
27576,acs-acs5,2021,C27005_001E,Estimate!!Total:,DIRECT-PURCHASE HEALTH INSURANCE BY SEX BY AGE,int,C27005,0,NaN,NaN,"C27005_001EA,C27005_001M,C27005_001MA",NaN
27597,acs-acs5,2021,C27006_001E,Estimate!!Total:,MEDICARE COVERAGE BY SEX BY AGE,int,C27006,0,NaN,NaN,"C27006_001EA,C27006_001M,C27006_001MA",NaN
27618,acs-acs5,2021,C27007_001E,Estimate!!Total:,MEDICAID/MEANS-TESTED PUBLIC COVERAGE BY SEX B...,int,C27007,0,NaN,NaN,"C27007_001EA,C27007_001M,C27007_001MA",NaN
27639,acs-acs5,2021,C27008_001E,Estimate!!Total:,TRICARE/MILITARY HEALTH COVERAGE BY SEX BY AGE,int,C27008,0,NaN,NaN,"C27008_001EA,C27008_001M,C27008_001MA",NaN


And we can continue filtering down some more

In [11]:
sxage.concept.unique()

<ArrowStringArray>
[                                                                                                     'SEX BY AGE (WHITE ALONE)',
                                                                                  'SEX BY AGE (BLACK OR AFRICAN AMERICAN ALONE)',
                                                                          'SEX BY AGE (AMERICAN INDIAN AND ALASKA NATIVE ALONE)',
                                                                                                      'SEX BY AGE (ASIAN ALONE)',
                                                                 'SEX BY AGE (NATIVE HAWAIIAN AND OTHER PACIFIC ISLANDER ALONE)',
                                                                                            'SEX BY AGE (SOME OTHER RACE ALONE)',
                                                                                                'SEX BY AGE (TWO OR MORE RACES)',
                                                                       

From this filtering, say we're interested in `'SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE POPULATION 18 YEARS AND OVER'` we can then filter our original dataset to get all of the variable codes that describe this information from the Census

In [12]:
sx_age_ed = acs_vars[acs_vars["concept"] == 'SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE POPULATION 18 YEARS AND OVER']
sx_age_ed

,dataset,year,name,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
8040,acs-acs5,2021,B15001_001E,Estimate!!Total:,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_001EA,B15001_001M,B15001_001MA",NaN
8041,acs-acs5,2021,B15001_002E,Estimate!!Total:!!Male:,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_002EA,B15001_002M,B15001_002MA",NaN
8042,acs-acs5,2021,B15001_003E,Estimate!!Total:!!Male:!!18 to 24 years:,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_003EA,B15001_003M,B15001_003MA",NaN
8043,acs-acs5,2021,B15001_004E,Estimate!!Total:!!Male:!!18 to 24 years:!!Less...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_004EA,B15001_004M,B15001_004MA",NaN
8044,acs-acs5,2021,B15001_005E,Estimate!!Total:!!Male:!!18 to 24 years:!!9th ...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_005EA,B15001_005M,B15001_005MA",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
8118,acs-acs5,2021,B15001_079E,Estimate!!Total:!!Female:!!65 years and over:!...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_079EA,B15001_079M,B15001_079MA",NaN
8119,acs-acs5,2021,B15001_080E,Estimate!!Total:!!Female:!!65 years and over:!...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_080EA,B15001_080M,B15001_080MA",NaN
8120,acs-acs5,2021,B15001_081E,Estimate!!Total:!!Female:!!65 years and over:!...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_081EA,B15001_081M,B15001_081MA",NaN
8121,acs-acs5,2021,B15001_082E,Estimate!!Total:!!Female:!!65 years and over:!...,SEX BY AGE BY EDUCATIONAL ATTAINMENT FOR THE P...,int,B15001,0,NaN,NaN,"B15001_082EA,B15001_082M,B15001_082MA",NaN


# Creating custom defaults

The Census Data Collector has classes to load and build data tables of defaults. First step is importing the Default variable handler

In [13]:
from censusdc.defaults import UserDefaults

defaults = UserDefaults()

Now we can begin saving our default variables from the Sex by Age by Eductation pull with an associated name/description. The `add_defaults()` method allows us to attach information to the `UserDefaults` object. The parameters for this method include:
   - parameter_code : census variable code (str)
   - name : user defined name for the variable (str)

In [14]:
for pcode, label in zip(sx_age_ed["name"], sx_age_ed["label"]):
    name = (label.split(":!!"))
    if len(name) < 2:
        continue
    name = " ".join(name[1:])
    name = name.replace(":", "")
    defaults.add_defaults(pcode, name)

defaults.dataframe

,name,cen_code
0,Male,B15001_002E
1,Male 18 to 24 years,B15001_003E
2,Male 18 to 24 years Less than 9th grade,B15001_004E
3,"Male 18 to 24 years 9th to 12th grade, no diploma",B15001_005E
4,Male 18 to 24 years High school graduate (incl...,B15001_006E
...,...,...
77,Female 65 years and over High school graduate ...,B15001_079E
78,"Female 65 years and over Some college, no degree",B15001_080E
79,Female 65 years and over Associate's degree,B15001_081E
80,Female 65 years and over Bachelor's degree,B15001_082E


We can also remove defaults. For example the total counts of males and females

In [15]:
# remove default by parameter code
defaults.remove_defaults(parameter_code="B15001_002E")

# remove default by name
defaults.remove_defaults(name="Female")

In [16]:
defaults.dataframe

,name,cen_code
0,Male 18 to 24 years,B15001_003E
1,Male 18 to 24 years Less than 9th grade,B15001_004E
2,"Male 18 to 24 years 9th to 12th grade, no diploma",B15001_005E
3,Male 18 to 24 years High school graduate (incl...,B15001_006E
4,"Male 18 to 24 years Some college, no degree",B15001_007E
...,...,...
75,Female 65 years and over High school graduate ...,B15001_079E
76,"Female 65 years and over Some college, no degree",B15001_080E
77,Female 65 years and over Associate's degree,B15001_081E
78,Female 65 years and over Bachelor's degree,B15001_082E


We can write this information to file using the `write_defaults()` method.

In [17]:
defaults.write_defaults("my_default_file.dat")

And we can load it back up for later use using the `.load()` method

In [18]:
defaults2 = UserDefaults.load("my_default_file.dat")
defaults2.dataframe

,name,cen_code
0,Male 18 to 24 years,B15001_003E
1,Male 18 to 24 years Less than 9th grade,B15001_004E
2,"Male 18 to 24 years 9th to 12th grade, no diploma",B15001_005E
3,Male 18 to 24 years High school graduate (incl...,B15001_006E
4,"Male 18 to 24 years Some college, no degree",B15001_007E
...,...,...
75,Female 65 years and over High school graduate ...,B15001_079E
76,"Female 65 years and over Some college, no degree",B15001_080E
77,Female 65 years and over Associate's degree,B15001_081E
78,Female 65 years and over Bachelor's degree,B15001_082E
